# Machine Unlearning

In [1]:
# import required libraries
import numpy as np
import tarfile
import os
import time

import torch
from torch import nn
import torch.nn.functional as F
from torchvision.datasets.utils import download_url
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as tt
from torchvision.models import resnet18
from torchvision import datasets

torch.manual_seed(100)

## Helper Functions

In [2]:
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

def training_step(model, batch):
    images, labels = batch
    images, labels = images.to(device), labels.to(device)
    out = model(images)                  
    loss = F.cross_entropy(out, labels) 
    return loss

def validation_step(model, batch):
    images, labels = batch
    images, labels = images.to(device), labels.to(device)
    out = model(images)                    
    loss = F.cross_entropy(out, labels)   
    acc = accuracy(out, labels)
    return {'Loss': loss.detach(), 'Acc': acc}

def validation_epoch_end(model, outputs):
    batch_losses = [x['Loss'] for x in outputs]
    epoch_loss = torch.stack(batch_losses).mean()   
    batch_accs = [x['Acc'] for x in outputs]
    epoch_acc = torch.stack(batch_accs).mean()      
    return {'Loss': epoch_loss.item(), 'Acc': epoch_acc.item()}

def epoch_end(model, epoch, result):
    print("Epoch [{}], last_lr: {:.5f}, train_loss: {:.4f}, val_loss: {:.4f}, val_acc: {:.4f}".format(
        epoch, result['lrs'][-1], result['train_loss'], result['Loss'], result['Acc']))

    
def distance(model,model0):
    distance=0
    normalization=0
    for (k, p), (k0, p0) in zip(model.named_parameters(), model0.named_parameters()):
        space='  ' if 'bias' in k else ''
        current_dist=(p.data0-p0.data0).pow(2).sum().item()
        current_norm=p.data0.pow(2).sum().item()
        distance+=current_dist
        normalization+=current_norm
    print(f'Distance: {np.sqrt(distance)}')
    print(f'Normalized Distance: {1.0*np.sqrt(distance/normalization)}')
    return 1.0*np.sqrt(distance/normalization)

In [3]:
@torch.no_grad()
def evaluate(model, val_loader):
    model.eval()
    outputs = [validation_step(model, batch) for batch in val_loader]
    return validation_epoch_end(model, outputs)

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit_one_cycle(epochs, max_lr, model, train_loader, val_loader, 
                  weight_decay=0, grad_clip=None, opt_func=torch.optim.SGD):
    torch.cuda.empty_cache()
    history = []
    
    optimizer = opt_func(model.parameters(), max_lr, weight_decay=weight_decay)

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    
    for epoch in range(epochs): 
        model.train()
        train_losses = []
        lrs = []
        for batch in train_loader:
            loss = training_step(model, batch)
            train_losses.append(loss)
            loss.backward()
            
            if grad_clip: 
                nn.utils.clip_grad_value_(model.parameters(), grad_clip)
            
            optimizer.step()
            optimizer.zero_grad()
            
            lrs.append(get_lr(optimizer))
            
        
        # Validation phase
        result = evaluate(model, val_loader)
        result['train_loss'] = torch.stack(train_losses).mean().item()
        result['lrs'] = lrs
        epoch_end(model, epoch, result)
        history.append(result)
        sched.step(result['Loss'])
    return history

## Train/Load the Model

### Cifar-10

In [4]:
batch_size = 256
# Define a transformation pipeline
transform_data = tt.Compose([
   # tt.Grayscale(num_output_channels=3),
    tt.Resize((64, 64)),  # Resize the image to the input size of the model
    tt.ToTensor(),         # Convert the image to a PyTorch tensor
    tt.Normalize(
        mean=[0.485, 0.456, 0.406],  # Normalize with the ImageNet mean values
        std=[0.229, 0.224, 0.225]    # Normalize with the ImageNet standard deviations
    ),
])


In [5]:
# #Dowload the dataset
# dataset_url = "https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz"
# download_url(dataset_url, '.')

# # Extract from archive
# with tarfile.open('./cifar10.tgz', 'r:gz') as tar:
#     tar.extractall(path='./data')
    
# # Look into the data directory
# data_dir = './data/cifar10'
# print(os.listdir(data_dir))
# classes = os.listdir(data_dir + "/train")
# print(classes)

# train_ds = ImageFolder(data_dir+'/train', transform_data)
# valid_ds = ImageFolder(data_dir+'/test', transform_data)

# VVG10

In [7]:
train_ds = ImageFolder('/kaggle/input/vggface/VGG10/train', transform_data)
valid_ds = ImageFolder('/kaggle/input/vggface/VGG10/test', transform_data)

### MNIST Digit

In [8]:
# train_ds = datasets.MNIST(
#     root='./data',        # Specify the directory to download the dataset to
#     train=True,            # Load the training dataset
#     transform=transform_data,   # Apply the defined transform to the data
#     download=True          # Download the dataset if it's not already downloaded
# )

# # Load the MNIST test dataset
# valid_ds = datasets.MNIST(
#     root='./data',
#     train=False,           # Load the test dataset
#     transform=transform_data,
#     download=True
# )

### FashionMNIST

In [9]:
# train_ds = datasets.FashionMNIST(
#     root='./data',        # Specify the directory to download the dataset to
#     train=True,            # Load the training dataset
#     transform=transform_data,   # Apply the defined transform to the data
#     download=True          # Download the dataset if it's not already downloaded
# )

# # Load the MNIST test dataset
# valid_ds = datasets.FashionMNIST(
#     root='./data',
#     train=False,           # Load the test dataset
#     transform=transform_data,
#     download=True
# )

In [10]:
train_dl = DataLoader(train_ds, batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_dl = DataLoader(valid_ds, batch_size*2, num_workers=2, pin_memory=True)

### Train and save the model

In [11]:
%%time
# Load a pre-trained ResNet model
resnet_model = resnet18(pretrained=True)

# Modify the last layer's activation function
num_classes = 10  # Change this to the number of classes in your problem
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

# Optionally, you can add a different activation function to the last layer
activation_function = nn.Sigmoid()  # Change this to the desired activation function
resnet_model.add_module("activation", activation_function)

device = "cuda:0"
model = resnet_model.to(device = device)

epochs = 100
max_lr = 0.001
grad_clip = 0.01
weight_decay = 1e-1
opt_func = torch.optim.Adam

# history = fit_one_cycle(epochs, max_lr, model, train_dl, valid_dl, 
#                              grad_clip=grad_clip, 
#                              weight_decay=weight_decay, 
#                              opt_func=opt_func)

# torch.save(model.state_dict(), "ResNET18_VGG10.pt")

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s] 


CPU times: user 1.4 s, sys: 896 ms, total: 2.3 s
Wall time: 4 s


### Testing the Model

In [12]:
model.load_state_dict(torch.load("/kaggle/input/models/ResNET18_VGG10.pt"))
history = [evaluate(model, valid_dl)]
history

[{'Loss': 0.6571452617645264, 'Acc': 0.8061151504516602}]

## Unlearning

In [13]:
# defining the noise structure
class Noise(nn.Module):
    def __init__(self, *dim):
        super().__init__()
        self.noise = torch.nn.Parameter(torch.randn(*dim), requires_grad = True)
        
    def forward(self):
        return self.noise

In [15]:
def get_data(classes_to_forget,train_ds):
    print("Making Dataset:")
    # list of all classes
    classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    %reset_selective -f classwise_train classwise_test
    # classwise list of samples
    num_classes = 10
    classwise_train = {}
    for i in range(num_classes):
        classwise_train[i] = []

    for img, label in train_ds:
        classwise_train[label].append((img, label))

    classwise_test = {}
    for i in range(num_classes):
        classwise_test[i] = []

    for img, label in valid_ds:
        classwise_test[label].append((img, label))
    
    # getting some samples from retain classes
    print("Getting some samples from retain classes:")
    num_samples_per_class = 1000

    retain_samples = []
    for i in range(len(classes)):
        if classes[i] not in classes_to_forget:
            retain_samples += classwise_train[i][:num_samples_per_class]
            
            
    # retain validation set
    print("Getting retain validation classes:")
    retain_valid = []
    for cls in range(num_classes):
        if cls not in classes_to_forget:
            for img, label in classwise_test[cls]:
                retain_valid.append((img, label))

    # forget validation set
    forget_valid = []
    for cls in range(num_classes):
        if cls in classes_to_forget:
            for img, label in classwise_test[cls]:
                forget_valid.append((img, label))

    # forget_valid_dl = DataLoader(forget_valid, batch_size, num_workers=2, pin_memory=True)
    forget_valid_dl = DataLoader(forget_valid, 1, num_workers=2, pin_memory=True)

    retain_valid_dl = DataLoader(retain_valid, batch_size*2, num_workers=2, pin_memory=True)
    
    
    return forget_valid_dl, retain_valid_dl,retain_samples
    

# Original

In [99]:
def get_accuracy(classes_to_forget,forget_valid_dl, retain_valid_dl,retain_samples,model_a,batch_size=100):
    
    noises = {}
    start_time = time.time()
    total_time = 0
    for cls in classes_to_forget:
        print("Optiming loss for class {}".format(cls))
        noises[cls] = Noise(batch_size, 3, 64, 64).cuda()
        opt = torch.optim.Adam(noises[cls].parameters(), lr = 0.1)

        num_epochs = 5
        num_steps = 8
        class_label = cls
        for epoch in range(num_epochs):
            total_loss = []
            for batch in range(num_steps):
                inputs = noises[cls]()
                labels = torch.zeros(batch_size).cuda()+class_label
                outputs = model_a(inputs)
                loss = -F.cross_entropy(outputs, labels.long()) + 0.009*torch.mean(torch.sum(torch.square(inputs), [1, 2, 3]))
                opt.zero_grad()
                loss.backward()
                opt.step()
                total_loss.append(loss.cpu().detach().numpy())
            print("Loss: {}".format(np.mean(total_loss)))

    print("Loading data from retain")   
    total_time += (time.time() - start_time)
    noisy_data = []
    num_batches = 20
    class_num = 0

    for cls in classes_to_forget:
        for i in range(num_batches):
            batch = noises[cls]().cpu().detach()
            for i in range(batch[0].size(0)):
                noisy_data.append((batch[i], torch.tensor(random.choice(classes_to_forget))))
    
    other_samples = []
    for i in range(len(retain_samples)):
        other_samples.append((retain_samples[i][0].cpu(), torch.tensor(retain_samples[i][1])))
        
    noisy_data += other_samples
    noisy_loader = torch.utils.data.DataLoader(noisy_data, batch_size=batch_size, shuffle = True)


    optimizer = torch.optim.Adam(model_a.parameters(), lr = 0.00009)
    start_time = time.time()
    print("Impairing")  
    for epoch in range(1):  
        model_a.train(True)
        running_loss = 0.0
        running_acc = 0
        for i, data in enumerate(noisy_loader):
            inputs, labels = data
            inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()

            optimizer.zero_grad()
            outputs = model_a(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item() * inputs.size(0)
            out = torch.argmax(outputs.detach(),dim=1)
            assert out.shape==labels.shape
            running_acc += (labels==out).sum().item()
        print(f"Train loss {epoch+1}: {running_loss/len(train_ds)},Train Acc:{running_acc*100/len(train_ds)}%")

    total_time += (time.time() - start_time)
#     print("Performance of Standard Forget Model on Forget Class")
#     history = [evaluate(model_a, forget_valid_dl)]
#     print("Accuracy: {}".format(history[0]["Acc"]*100))
#     print("Loss: {}".format(history[0]["Loss"]))

#     print("Performance of Standard Forget Model on Retain Class")
#     history = [evaluate(model_a, retain_valid_dl)]
#     print("Accuracy: {}".format(history[0]["Acc"]*100))
#     print("Loss: {}".format(history[0]["Loss"]))    
    
    heal_loader = torch.utils.data.DataLoader(other_samples, batch_size=batch_size, shuffle = True)

    optimizer = torch.optim.Adam(model_a.parameters(), lr = 0.0001)

    print("Repairing")
    start_time = time.time()
    for epoch in range(1):  
        model_a.train(True)
        running_loss = 0.0
        running_acc = 0
        for i, data in enumerate(heal_loader):
            inputs, labels = data
            inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()

            optimizer.zero_grad()
            outputs = model_a(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item() * inputs.size(0)
            out = torch.argmax(outputs.detach(),dim=1)
            assert out.shape==labels.shape
            running_acc += (labels==out).sum().item()
        print(f"Train loss {epoch+1}: {running_loss/len(train_ds)},Train Acc:{running_acc*100/len(train_ds)}%")

    total_time += (time.time() - start_time)
#     print("Prediction on D_f:\n")
#     for i, data in enumerate(forget_valid_dl):
#             model_a.eval()
#             inputs, labels = data
#             inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()
#             outputs = model_a(inputs)
#             print(outputs,"\n","-"*100)
#             if i==5:
#                 break

    print("Performance of Standard Forget Model on Forget Class")
    history = [evaluate(model_a, forget_valid_dl)]
    print("Accuracy DF(O): {}".format(history[0]["Acc"]*100))
    print("Loss: {}".format(history[0]["Loss"]))

    print("Performance of Standard Forget Model on Retain Class")
    history = [evaluate(model_a, retain_valid_dl)]
    print("Accuracy DR(O): {}".format(history[0]["Acc"]*100))
    print("Loss: {}".format(history[0]["Loss"]))  
    print(f"Total T(O):{total_time}")

    

In [100]:
model_a = resnet18(num_classes = 10).to(device = device)
model_a.load_state_dict(torch.load("/kaggle/input/models/ResNET18_VGG10.pt"))

get_accuracy(classes_to_forget=classes_to_forget,
             forget_valid_dl=forget_valid_dl,
             retain_valid_dl=retain_valid_dl,
             retain_samples=retain_samples,
             model_a=model_a,
            batch_size=100)

Optiming loss for class 0
Loss: 70.69404602050781
Loss: 19.450254440307617
Loss: 2.7478694915771484
Loss: -2.092519760131836
Loss: -3.976229667663574
Loading data from retain
Impairing


/tmp/ipykernel_32/3047033246.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.11106153957583835,Train Acc:89.06303236797274%
Repairing


/tmp/ipykernel_32/3047033246.py:94: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.13531403668491357,Train Acc:85.58773424190801%
Performance of Standard Forget Model on Forget Class
Accuracy DF(O): 0.0
Loss: 4.882045745849609
Performance of Standard Forget Model on Retain Class
Accuracy DR(O): 74.66917634010315
Loss: 1.163283109664917
Total T(O):3.6254308223724365


# My

In [119]:
import random
def get_accuracy_ma(classes_to_forget,forget_valid_dl, retain_valid_dl,retain_samples,model_m,batch_size=100):
    
    start_time = time.time()
    total_time = 0
   
    noise = Noise(batch_size, 3, 64, 64).cuda()
    opt = torch.optim.Adam(noise.parameters(), lr = 0.1)

    num_epochs = 1
    num_steps = 8

    for epoch in range(num_epochs):
        total_loss = []
        for batch in range(num_steps):
            inputs = noise()
            labels = torch.tensor([random.choice(classes_to_forget) for _ in range(batch_size)]).cuda()

            outputs = model_m(inputs)

            loss = - F.cross_entropy(outputs, labels.long()) + 0.01 * torch.mean(torch.sqrt(torch.sum(labels - outputs.T)))

            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss.append(loss.cpu().detach().numpy())

        print("Loss: {}".format(np.mean(total_loss)))
    print("Loading data from retain")   
    total_time += (time.time() - start_time)
   
    noisy_data = []
    num_batches = 20


    for i in range(num_batches):
        batch = noise().cpu().detach()
        for i in range(batch[0].size(0)):
            noisy_data.append((batch[i], torch.tensor(random.choice(classes_to_forget))))

    other_samples = []
    for i in range(len(retain_samples)):
        other_samples.append((retain_samples[i][0].cpu(), torch.tensor(retain_samples[i][1])))
    noisy_data += other_samples
    noisy_loader = torch.utils.data.DataLoader(noisy_data, batch_size=batch_size, shuffle = True)


    optimizer = torch.optim.Adam(model_m.parameters(), lr = 0.00009)
    start_time = time.time()
    print("Impairing")  
    for epoch in range(1):  
        model_m.train(True)
        running_loss = 0.0
        running_acc = 0
        for i, data in enumerate(noisy_loader):
            inputs, labels = data
            inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()

            optimizer.zero_grad()
            outputs = model_m(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item() * inputs.size(0)
            out = torch.argmax(outputs.detach(),dim=1)
            assert out.shape==labels.shape
            running_acc += (labels==out).sum().item()
        print(f"Train loss {epoch+1}: {running_loss/len(train_ds)},Train Acc:{running_acc*100/len(train_ds)}%")

    total_time += (time.time() - start_time)
#     print("Performance of Standard Forget Model on Forget Class")
#     history = [evaluate(model_a, forget_valid_dl)]
#     print("Accuracy of Forget Orig MY: {}".format(history[0]["Acc"]*100))
#     print("Loss: {}".format(history[0]["Loss"]))

#     print("Performance of Standard Forget Model on Retain Class")
#     history = [evaluate(model_a, retain_valid_dl)]
#     print("Accuracy: {}".format(history[0]["Acc"]*100))
#     print("Loss: {}".format(history[0]["Loss"]))    
    
    heal_loader = torch.utils.data.DataLoader(other_samples, batch_size=batch_size, shuffle = True)

    optimizer = torch.optim.Adam(model_m.parameters(), lr = 0.0001)

    print("Repairing")
    start_time = time.time()
    for epoch in range(1):  
        model_m.train(True)
        running_loss = 0.0
        running_acc = 0
        for i, data in enumerate(heal_loader):
            inputs, labels = data
            inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()

            optimizer.zero_grad()
            outputs = model_m(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item() * inputs.size(0)
            out = torch.argmax(outputs.detach(),dim=1)
            assert out.shape==labels.shape
            running_acc += (labels==out).sum().item()
        print(f"Train loss {epoch+1}: {running_loss/len(train_ds)},Train Acc:{running_acc*100/len(train_ds)}%")

    total_time += (time.time() - start_time)
#     print("Prediction on D_f:\n")
#     for i, data in enumerate(forget_valid_dl):
#             model_m.eval()
#             inputs, labels = data
#             inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()
#             outputs = model_m(inputs)
#             print(outputs,"\n","-"*100)
#             if i==5:
#                 break

    print("Performance of Standard Forget Model on Forget Classes")
    history = [evaluate(model_m, forget_valid_dl)]
    print("Accuracy DF(M): {}".format(history[0]["Acc"]*100))
    print("Loss: {}".format(history[0]["Loss"]))

    print("Performance of Standard Forget Model on Retain Class")
    history = [evaluate(model_m, retain_valid_dl)]
    print("Accuracy DR(M): {}".format(history[0]["Acc"]*100))
    print("Loss: {}".format(history[0]["Loss"]))  
    print(f"Total T(M):{total_time}")

    

In [120]:
model_m = resnet18(num_classes = 10).to(device = device)
model_m.load_state_dict(torch.load("/kaggle/input/models/ResNET18_VGG10.pt"))

get_accuracy_ma(classes_to_forget=classes_to_forget,
             forget_valid_dl=forget_valid_dl,
             retain_valid_dl=retain_valid_dl,
             retain_samples=retain_samples,
             model_m=model_m,
            batch_size=100)

Loss: -6.139523983001709
Loading data from retain
Impairing


/tmp/ipykernel_32/3917435335.py:57: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.10423301369510274,Train Acc:89.3015332197615%
Repairing


/tmp/ipykernel_32/3917435335.py:95: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.12226684353674494,Train Acc:86.37137989778535%
Performance of Standard Forget Model on Forget Classes
Accuracy DF(M): 0.0
Loss: 4.411697864532471
Performance of Standard Forget Model on Retain Class
Accuracy DR(M): 66.48927330970764
Loss: 1.380441665649414
Total T(M):2.412562370300293


In [121]:
def get_acc_weight_update(classes_to_forget,model_s,retain_valid_dl):
    u_w = torch.zeros(size=(512,))
    start_time = time.time()
    for i in classes_to_forget:
        model_s.fc.weight.data[i] = u_w  # Replace 'model.fc' with your output layer
        print(i)
    total_time =(time.time() - start_time)
    print("Performance of Standard Forget Model on Forget Class")
    history = [evaluate(model_s, forget_valid_dl)]
    print("Accuracy DF(W): {}".format(history[0]["Acc"]*100))
    print("Loss: {}".format(history[0]["Loss"]))

    print("Performance of Standard Forget Model on Retain Class")
    history = [evaluate(model_s, retain_valid_dl)]
    print("Accuracy DR(W): {}".format(history[0]["Acc"]*100))
    print("Loss: {}".format(history[0]["Loss"]))   
    print(f"Total Time(W):{total_time}")



In [122]:
# Define a replacement dictionary for each scenario
replacement_dict_1 = {0: 1, 1: 2, 2: 3, 3:4, 4:5, 5:6, 6:7, 7:8, 8:9}
replacement_dict_2 = {6: 7, 7: 8, 8: 9}
replacement_dict_3 = {1: 2, 2:4, 3: 5, 4:6, 5:7, 6:8, 7:9}
replacement_dict_4 = {4: 6, 5: 8, 6: 9}
replacement_dict_5 = {1: 3, 2: 4, 3: 6}
def replace_elements(A, replacement_dict):
    return [replacement_dict.get(x, x) for x in A]
A = [0,1,2,3]
replace_elements(A, replacement_dict_5)


[0, 3, 4, 6]

In [123]:
def calculate_accuracy(model, data_loader,  replacement_dict):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient tracking for evaluation
        for batch in data_loader:
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)  # Move data to the device (CPU or GPU)

            # Forward pass
            outputs = model(inputs)

            # Get the predicted classes
            _, predicted = torch.max(outputs, 1)
            # Replace the forgotten classes in predicted with the original indices
            print(f"Before Replacement:{torch.unique(predicted)}")
            predicted_rep = torch.tensor(replace_elements(predicted.tolist(),replacement_dict),device="cuda:0")
            
            print(f"After Replacement:{torch.unique(predicted_rep)}")

            # Update the total and correct predictions
            total += labels.size(0)
            correct += (predicted_rep == labels).sum().item()

    # Calculate accuracy
    accuracy = 100 * (correct / total)
    return accuracy


In [124]:
import time

def get_acc_weight_delete(classes_to_forget, model_d, num_classes,retain_valid_dl,replacement_dict):
    start_time = time.time()

    # Calculate the number of output classes after deleting specified classes
    new_num_classes = num_classes - len(classes_to_forget)

    # Create a new fully connected layer with the correct number of output units
    new_fc = nn.Linear(in_features=model_d.fc.in_features, out_features=new_num_classes, bias=True)

    # Initialize indices to keep track of the remaining classes
    remaining_indices = [i for i in range(num_classes) if i not in classes_to_forget]
    # Initialize new weight and bias tensors
    new_weight = model_d.fc.weight.data[remaining_indices,:]
    new_bias = model_d.fc.bias.data[remaining_indices]    
    # Copy the new weight and bias tensors to the new fully connected layer
    new_fc.weight.data = new_weight
    new_fc.bias.data = new_bias
    # Replace the original fully connected layer with the new one
    model_d.fc = new_fc
   
    total_time = time.time() - start_time
   

    # Evaluate the model with the modified architecture

    print("Performance of Standard Forget Model on Retain Class")
#     history = [evaluate(model_d, retain_valid_dl)]
#     print("Accuracy DR(W): {}".format(history[0]["Acc"]*100))
#     print("Loss: {}".format(history[0]["Loss"]))   
    print(f"Accuracy:{calculate_accuracy(model_d, retain_valid_dl,replacement_dict=replacement_dict)}")
    print(f"Total Time(W):{total_time}")


In [125]:
import gc

In [131]:
gc.collect()

model_a = resnet18(num_classes = 10).to(device = device)
model_a.load_state_dict(torch.load("/kaggle/input/models/ResNET18_VGG10.pt"))

model_m = resnet18(num_classes = 10).to(device = device)
model_m.load_state_dict(torch.load("/kaggle/input/models/ResNET18_VGG10.pt"))

model_w = resnet18(num_classes = 10).to(device = device)
model_w.load_state_dict(torch.load("/kaggle/input/models/ResNET18_VGG10.pt"))

model_d = resnet18(num_classes = 10).to(device = device)
model_d.load_state_dict(torch.load("/kaggle/input/models/ResNET18_VGG10.pt"))

classes_to_forget = [1,2,5,7,8,9]


forget_valid_dl, retain_valid_dl,retain_samples= get_data(classes_to_forget=classes_to_forget,train_ds=train_ds)

print("get_acc_author_method")
get_accuracy(classes_to_forget=classes_to_forget,
             forget_valid_dl=forget_valid_dl,
             retain_valid_dl=retain_valid_dl,
             retain_samples=retain_samples,
             model_a=model_a,
            batch_size=100)

print("-"*30,"\nget_acc_my_method")
get_accuracy_ma(classes_to_forget=classes_to_forget,
             forget_valid_dl=forget_valid_dl,
             retain_valid_dl=retain_valid_dl,
             retain_samples=retain_samples,
             model_m=model_m,
            batch_size=100)

print("-"*30,"\nget_acc_weight_update")
get_acc_weight_update(classes_to_forget,model_w,retain_valid_dl)

print("-"*30,"\nget_acc_weight_delete")
get_acc_weight_delete(classes_to_forget=classes_to_forget,model_d=model_d,num_classes=10,
                      retain_valid_dl=retain_valid_dl,replacement_dict=replacement_dict_5)


Making Dataset:
Getting some samples from retain classes:
Getting retain validation classes:
get_acc_author_method
Optiming loss for class 1
Loss: 69.65628814697266
Loss: 18.557083129882812
Loss: 2.6227426528930664
Loss: -1.6187307834625244
Loss: -3.4586970806121826
Optiming loss for class 2
Loss: 70.17117309570312
Loss: 18.87847900390625
Loss: 2.5376384258270264
Loss: -2.2838454246520996
Loss: -4.243780136108398
Optiming loss for class 5
Loss: 70.3553237915039
Loss: 19.1343994140625
Loss: 2.526378631591797
Loss: -2.2798566818237305
Loss: -3.9415249824523926
Optiming loss for class 7
Loss: 69.3387222290039
Loss: 18.243179321289062
Loss: 2.213965892791748
Loss: -2.408801555633545
Loss: -4.279851913452148
Optiming loss for class 8
Loss: 69.39642333984375
Loss: 17.72347068786621
Loss: 1.894716739654541
Loss: -2.526747465133667
Loss: -4.27540922164917
Optiming loss for class 9
Loss: 71.29452514648438
Loss: 20.188220977783203
Loss: 3.273908853530884
Loss: -1.7976899147033691
Loss: -3.970146

/tmp/ipykernel_32/3047033246.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.3839222678943225,Train Acc:35.1618398637138%
Repairing


/tmp/ipykernel_32/3047033246.py:94: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.10033235970307046,Train Acc:32.947189097103916%
Performance of Standard Forget Model on Forget Class
Accuracy DF(O): 1.4354066923260689
Loss: 3.035550355911255
Performance of Standard Forget Model on Retain Class
Accuracy DR(O): 77.44361162185669
Loss: 0.9563596844673157
Total T(O):9.742892265319824
------------------------------ 
get_acc_my_method
Loss: -5.503129959106445
Loading data from retain
Impairing


/tmp/ipykernel_32/3917435335.py:57: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.18678191845421085,Train Acc:33.69676320272573%
Repairing


/tmp/ipykernel_32/3917435335.py:95: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs, labels = inputs.cuda(),torch.tensor(labels).cuda()


Train loss 1: 0.07218275604296787,Train Acc:33.3901192504259%
Performance of Standard Forget Model on Forget Classes
Accuracy DF(M): 0.0
Loss: 3.5846598148345947
Performance of Standard Forget Model on Retain Class
Accuracy DR(M): 62.03007698059082
Loss: 1.600272297859192
Total T(M):1.1419472694396973
------------------------------ 
get_acc_weight_update
1
2
5
7
8
9
Performance of Standard Forget Model on Forget Class
Accuracy DF(W): 16.267941892147064
Loss: 2.1882989406585693
Performance of Standard Forget Model on Retain Class
Accuracy DR(W): 88.345867395401
Loss: 0.48795512318611145
Total Time(W):0.0003445148468017578
------------------------------ 
get_acc_weight_delete
Performance of Standard Forget Model on Retain Class
Before Replacement:tensor([0, 1, 2, 3], device='cuda:0')
After Replacement:tensor([0, 3, 4, 6], device='cuda:0')
Accuracy:89.09774436090225
Total Time(W):0.0006721019744873047
